# 3. Default Definition Test

| | |
|---|---|
| **Goal** | Verify that the code correctly applies the adopted definition |
| **Input Data** | `default/` from Notebook 2 |
| **Result** | test tables in `reports/` |

## Three tests

1. **Artificial credits.** Comparison of code results.
2. **Independent calculation.** The same calculation using different data.
3. **List of conditions.** Verification that the results meet the conditions adopted based on the CRR.

**Important:** This test demonstrates the code's compliance with the adopted definition. It does not prove full compliance with the CRR, as some requirements cannot be verified with this data - list at the end.

In [1]:
import polars as pl

import config as C
import functions as F

pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(20)
pl.Config.set_fmt_str_lengths(70)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_hide_column_data_types(True)
C.REPORT_DIR.mkdir(parents=True, exist_ok=True)

## Test 1. Artificial Credits

Each rule should have a PASS result.

In [2]:
examples = F.rule_examples()
examples.write_csv(C.REPORT_DIR / "rule_examples.csv")
examples

rule,expected_default,result,reason,test
"""Two thresholds: one payment is not enough, default at status 4""","""2019-05""","""2019-05""","""DPD""","""PASS"""
"""Only the relative threshold exceeded: no default""",null,null,null,"""PASS"""
"""Only the absolute threshold exceeded: no default""",null,null,null,"""PASS"""
"""A repayment resets the counter""","""2019-06""","""2019-06""","""DPD""","""PASS"""
"""Status 2 for several months: default only at status 3""","""2019-05""","""2019-05""","""DPD""","""PASS"""
"""A loss sale is a default""","""2019-03""","""2019-03""","""UTP_loss_sale""","""PASS"""
"""A modification is a default""","""2019-02""","""2019-02""","""UTP_modification""","""PASS"""
"""COVID forbearance freezes the counter""","""2020-12""","""2020-12""","""DPD""","""PASS"""
"""Masked balance: the overdue amount can still be calculated, default at…","""2019-03""","""2019-03""","""DPD""","""PASS"""


## Test 2. Independent calculation

Random selection loans with and without defaults.

In [3]:
panel = pl.scan_parquet(C.DEFAULT_DIR / "*.parquet")
chosen = F.choose_test_loans(panel)
test = panel.filter(pl.col("LOAN_ID").is_in(chosen)).sort("LOAN_ID", "ACT_PERIOD").collect()

n_compared, n_different = F.independent_check(test)
print(f"Loans compared: {n_compared}")
print(f"Differences: {n_different}")

Loans compared: 3899
Differences: 0


## Test 3. List of Conditions

Each condition should have 0 violations.

In [4]:
checks = F.crr_checks(test, n_compared, n_different)
checks.write_csv(C.REPORT_DIR / "crr_test_results.csv")
checks

requirement,basis,violations,checked,result
"""The independent recalculation gives the same date and reason""","""method""",0,3899,"""PASS"""
"""A DPD default has a counter of at least 3""","""Art. 178(1)(b)""",0,1346,"""PASS"""
"""A DPD default has a status of at least 3 (90 days)""","""Art. 178(1)(b)""",0,1346,"""PASS"""
"""Both thresholds are exceeded in the DPD default month""","""Regulation 2018/171""",0,1346,"""PASS"""
"""A loss sale is a default""","""Art. 178(3)(c)""",0,644,"""PASS"""
"""A modification is a default""","""Art. 178(3)(d)""",0,44451,"""PASS"""
"""No DPD default in a COVID forbearance month""","""EBA/GL/2020/02""",0,1346,"""PASS"""
"""A loan has at most one default""","""study assumption""",0,3899,"""PASS"""
"""Months after default are marked correctly""","""study assumption""",0,240621,"""PASS"""


## Impact of Thresholds on Default
Comparison of the DPD default with a rule: the first month with a status of 3 or higher.

- **same month:** the thresholds did not change the date;
- **later:** the thresholds or forbearance moved the default;
- **no DPD default:** the loan reached status 3 but did not meet the DPD definition.

In [5]:
comparison = F.compare_with_simple_rule(panel)
comparison.write_csv(C.REPORT_DIR / "threshold_vs_simple_rule.csv")
comparison

result,loans,with_forbearance
"""later""",1063,203
"""no DPD default""",417,358
"""same month""",294,23


## Areas without test

| Requirement | Why |
|---|---|
| Exact number of days of delay | data is monthly |
| Exact arrears amount | Approximate of the amount based on the installment, excluding taxes and insurance |
| Threshold calculated for the debtor | data does not combine loans from the same person |
| Reduction in value by more than 1% during restructuring | modification is treated as default without this measurement |
| Other UTP criteria, e.g., bankruptcy | are not included in the data |
